In [2]:
!nvidia-smi

Thu Sep 17 15:27:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
from google.colab import drive
import os

drive.mount('/content/drive')

os.environ["HF_HOME"] = "/content/drive/MyDrive/vllm_cache"
print("Đã thiết lập thư mục lưu model tại Google Drive!")


Mounted at /content/drive
Đã thiết lập thư mục lưu model tại Google Drive!


In [5]:
!pip install -q vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/

In [6]:
import vllm
print("Cài đặt thành công! Phiên bản vLLM:", vllm.__version__)

Cài đặt thành công! Phiên bản vLLM: 0.29.0


In [7]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


## Cách gọi model qua file

In [ ]:
%%writefile test_vllm.py
from vllm import LLM, SamplingParams

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.9,
    max_tokens=256
)

llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    quantization="awq",                   # Khai báo rõ thuật toán nén 4-bit AWQ
    dtype="half",                         # Bắt buộc cho T4
    gpu_memory_utilization=0.85,          # Chiếm 85% VRAM của T4
    max_model_len=4096,                   # Nhận ngữ cảnh lên đến 4096 tokens
    enforce_eager=True
)

prompts = ["Giải thích ngắn gọn vLLM là gì trong 2 câu."]
outputs = llm.generate(prompts, sampling_params)

for output in outputs:
    print(f"\n[Kết quả]: {output.outputs[0].text}")


Writing test_vllm.py


In [ ]:
!python /content/test_vllm.py

INFO 09-17 02:01:22 [api_utils.py:286] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-17 02:01:22 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_USE_V1
INFO 09-17 02:01:24 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-17 02:01:24 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-17 02:01:24 [model.py:2021] Using max model len 2048
INFO 09-17 02:01:24 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 09-17 02:01:25 [vllm.py:1371] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-17 02:01:25 [vllm.py:1406] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 09-17 02:01:25 [kernel

# Cách gọi thẳng model

In [ ]:
"""[Câu hỏi dạng Text]
       │
       ▼ (1. Tokenizer)
[Dãy số Token IDs: [1524, 8821, ...]]
       │
       ▼ (2. Scheduler - Continuous Batching)
Gom các câu hỏi đang chờ vào chung 1 mẻ (Batch)
       │
       ▼ (3. Prefill Phase - Triton Attention)
Đọc toàn bộ câu hỏi 1 lần duy nhất, tính toán và lưu KV Cache vào bộ nhớ LBNHC
       │
       ▼ (4. Decode Phase - Sinh từng từ)
Lặp lại: Lấy từ trước + Đọc LBNHC KV Cache ──> Đoán từ tiếp theo
       │
       ▼ (5. Detokenizer)
Đổi các con số ngược lại thành chữ tiếng Việt trả về màn hình cho bạn
"""


In [2]:
import sys
from ipykernel.iostream import OutStream
OutStream.fileno = lambda self: 1

from vllm import LLM

llm = LLM(
    model="Qwen/Qwen2.5-7B-Instruct-AWQ",
    quantization="awq",
    dtype="half", #xu ly fallback
    gpu_memory_utilization=0.85,
    max_model_len=4096,
    enforce_eager=True
)

INFO 09-17 15:37:15 [api_utils.py:286] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 09-17 15:37:43 [model.py:684] Resolved architecture: Qwen2ForCausalLM
INFO 09-17 15:37:43 [model.py:2021] Using max model len 4096
INFO 09-17 15:37:44 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json:   0%|          | 0.00/62.7k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

WARNING 09-17 15:37:45 [vllm.py:1371] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 09-17 15:37:45 [vllm.py:1406] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 09-17 15:37:45 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
INFO 09-17 15:37:45 [vllm.py:1585] Cudagraph is disabled under eager mode
INFO 09-17 15:37:45 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


... (4 outputs left out)

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

(EngineCore pid=4182) INFO 09-17 15:37:53 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='Qwen/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=auto_awq, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, ot

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=4182) INFO 09-17 15:39:49 [default_loader.py:430] Loading weights took 17.71 seconds
(EngineCore pid=4182) INFO 09-17 15:39:53 [model_runner.py:404] Model loading took 5.29 GiB memory and 115.339670 seconds
(EngineCore pid=4182) WARNING 09-17 15:39:53 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=4182) INFO 09-17 15:39:53 [utils.py:306] Using LBNHC KV cache layout.
(EngineCore pid=4182) INFO 09-17 15:40:03 [gpu_worker.py:625] Available KV cache memory: 5.73 GiB
(EngineCore pid=4182) INFO 09-17 15:40:03 [kv_cache_utils.py:2032] GPU KV cache size: 107,344 tokens, Maximum concurrency for 4,096 tokens per request: 26.21x
(EngineCore pid=4182) INFO 09-17 15:40:04 [kernel_warmup.py:124] JIT kernel warmup starting.
(EngineCore pid=4182) INFO 09-17 15:40:04 [kernel_warmup.py:134] JIT kernel warmup finished in 0.00s.
(EngineCore pid=4182) INFO 

In [ ]:
from vllm import SamplingParams

sampling_params = SamplingParams(
    temperature=0.3,
    top_p=0.85,
    max_tokens=1024
)

conversations = [
    [
        {"role": "system", "content": "Bạn là chuyên gia AI. Trả lời hoàn toàn bằng tiếng Việt, không dùng tiếng Trung."},
        {"role": "user", "content": "Giải thích ngắn gọn vLLM là gì và các thành phần cốt lõi của nó."}
    ],
    [
        {"role": "system", "content": "Bạn là chuyên gia AI. Trả lời hoàn toàn bằng tiếng Việt, không dùng tiếng Trung."},
        {"role": "user", "content": "So sánh chi tiết sự khác nhau về kiến trúc giữa Qwen2.5-1.5B và Qwen2.5-7B."}
    ]
]

outputs = llm.chat(conversations, sampling_params)

for i, output in enumerate(outputs):
    print("\n" + "="*60)
    print(f"Câu hỏi {i+1}: {conversations[i][1]['content']}")
    print(f"Trả lời:\n{output.outputs[0].text.strip()}")


Rendering conversations:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 2/2 [00:22<00:00, 11.10s/it, est. speed input: 5.54 toks/s, output: 38.43 toks/s]


Câu hỏi 1: Giải thích ngắn gọn vLLM là gì và các thành phần cốt lõi của nó.
Trả lời:
vLLM là một mô hình ngôn ngữ lớn được phát triển bởi Học viện AI Thượng Hải. Nó được thiết kế để xử lý hiệu quả các tác vụ liên quan đến ngôn ngữ như dịch thuật, viết bài tự động và phân tích văn bản. Các thành phần cốt lõi của vLLM bao gồm:

1. **Thiết kế hiệu năng**: vLLM được tối ưu hóa để chạy nhanh trên các hệ thống có nhiều GPU, giúp xử lý các tác vụ ngôn ngữ một cách nhanh chóng và hiệu quả.

2. **Kiểm soát độ dài câu**: vLLM có khả năng kiểm soát độ dài của câu trả lời, giúp tránh việc tạo ra các câu trả lời quá dài hoặc không liên quan.

3. **Thiết lập độ chính xác**: vLLM được thiết kế để có thể điều chỉnh độ chính xác của kết quả dựa trên nhu cầu cụ thể của người dùng.

4. **Tích hợp các công nghệ tiên tiến**: vLLM sử dụng các công nghệ tiên tiến như注意力机制和自回归模型，以提高模型的理解能力和生成能力。这些技术帮助模型更好地理解输入的文本，并生成更自然和准确的输出。

5. **Dễ sử dụng**: vLLM cung cấp các API dễ sử dụng để tích hợp vào các ứng dụng 

# Đo tốc độ sinh token

In [3]:
import time
from vllm import SamplingParams

# 1. Định nghĩa tham số cấu hình
sampling_params = SamplingParams(
    temperature=0.3,
    top_p=0.85,
    max_tokens=512  # Giới hạn số token tối đa sinh ra để đo
)

# 2. Định nghĩa Prompt thử nghiệm
prompt = "Viết một bài văn ngắn khoảng 150 từ ca ngợi vẻ đẹp của Vịnh Hạ Long bằng tiếng Việt."

print("Đang gửi prompt tới vLLM và tính toán tốc độ...")
time_start = time.time()

# 3. Thực hiện Generation
outputs = llm.generate([prompt], sampling_params)

time_end = time.time()
elapsed_time = time_end - time_start

# 4. Trích xuất kết quả và tính toán số token
output = outputs[0]
generated_text = output.outputs[0].text
# Đếm số token được tạo ra thực tế trong output
num_generated_tokens = len(output.outputs[0].token_ids)

tokens_per_second = num_generated_tokens / elapsed_time

# 5. In kết quả đo lường
print("="*60)
print(f"[Kết quả sinh chữ]:\n{generated_text.strip()}")
print("="*60)
print(f"[Thống kê hiệu năng trên T4]:")
print(f"- Số lượng token đã sinh (Output Tokens): {num_generated_tokens} tokens")
print(f"- Thời gian xử lý tổng cộng: {elapsed_time:.2f} giây")
print(f"- Tốc độ sinh chữ sơ bộ: {tokens_per_second:.2f} tokens/giây (tok/s)")
print("="*60)

Đang gửi prompt tới vLLM và tính toán tốc độ...


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:23<00:00, 23.27s/it, est. speed input: 1.12 toks/s, output: 22.00 toks/s]

[Kết quả sinh chữ]:
Vịnh Hạ Long là một kỳ quan thiên nhiên tuyệt vời, nằm ở tỉnh Quảng Ninh, Việt Nam. Nơi đây được UNESCO công nhận là Di sản Thế giới vào năm 1999. Vịnh Hạ Long không chỉ nổi tiếng với hàng nghìn hòn đảo đá vôi độc đáo, mà còn với cảnh quan thiên nhiên hùng vĩ, từ những bãi cát trắng mịn đến rừng tre xanh mướt. Khung cảnh nơi đây thay đổi theo mùa, từ sắc xanh của mùa xuân đến sắc vàng của mùa thu. Ban đêm, ánh đèn của các hòn đảo tạo nên một bức tranh lung linh, thu hút du khách từ khắp nơi trên thế giới. Vịnh Hạ Long không chỉ là một điểm du lịch hấp dẫn, mà còn là một kho tàng văn hóa và lịch sử, chứa đựng nhiều câu chuyện cổ tích và truyền thuyết hấp dẫn. Vịnh Hạ Long là một món quà quý giá từ thiên nhiên, là biểu tượng của vẻ đẹp hoang sơ và độc đáo của Việt Nam. 

Bài văn này đã mô tả ngắn gọn vẻ đẹp của Vịnh Hạ Long từ nhiều góc độ khác nhau, từ cảnh quan thiên nhiên đến giá trị văn hóa lịch sử, đồng thời nhấn mạnh tầm quan trọng của việc bảo vệ và phát huy gi